# Real PhysioNet Dataset Validation & GAN Augmentation Study

This notebook demonstrates the complete, end-to-end Deep Learning pipeline to classify heart sound recordings into **Normal** and **Abnormal** classes using the official **PhysioNet Heart Sound Dataset (CinC Challenge 2016)**.

We investigate the effect of data augmentation by performing a comparison study:
1. **Baseline Model (No GAN)**: Trained on the real, unbalanced training data split.
2. **GAN-Augmented Model**: Trained on the training data balanced by generating synthetic abnormal MFCC features using a **1D DCGAN**.

To ensure scientific validity and prevent data leakage, train/validation/test splitting is performed at the recording level *before* segmentation.

### Step 1: Install & Import Dependencies

In [ ]:
raw_dir = 'data/raw'
download_dataset_subset(dest_dir=raw_dir, num_normal=117, num_abnormal=80)

### Step 2: Download Real PhysioNet Challenge Subset
We download 197 real heart sound recordings (117 normal, 80 abnormal) from PhysioNet training-a dataset directory and write `reference.csv`.

In [ ]:
raw_dir = 'data/raw'
download_dataset_subset(dest_dir=raw_dir, num_normal=117, num_abnormal=80)

### Step 3: Run Leak-proof Preprocessing & Feature Extraction
Performs stratified train/validation/test split at the recording level (70/15/15), runs overlap validation checks, then segments and extracts 13 MFCC coefficients for each split separately.

In [ ]:
processed_dir = 'data/processed'
process_dataset(raw_dir=raw_dir, processed_dir=processed_dir, fs=2000)

### Step 4: Exploratory Data Analysis (EDA)
Visualizing raw wave signals and MFCC heatmaps from the real PhysioNet dataset (Normal vs. Abnormal).

In [ ]:
print("Visualizing Real Normal Recording:")
plot_waveform_and_mfcc(os.path.join(raw_dir, 'a0007.wav'))

print("Visualizing Real Abnormal Recording:")
plot_waveform_and_mfcc(os.path.join(raw_dir, 'a0001.wav'))

### Step 5: Train the 1D GAN Model
We train the 1D DCGAN model on the real abnormal training segments only to learn their characteristic temporal patterns.
*Note: We run for 15 epochs for validation. Set to 50+ epochs for full training.*

In [ ]:
train_gan_pipeline(epochs=15, batch_size=32, latent_dim=100, processed_dir=processed_dir)

### Step 6: Train Baseline and GAN-Augmented Classifiers
Trains the baseline hybrid Conv1D-LSTM model on unbalanced training features, then uses the trained GAN generator to synthesize abnormal samples, balances classes, and trains the GAN-augmented model.
*Note: We run for 10 epochs for demonstration. Set to 20+ epochs for optimal performance.*

In [ ]:
train_classifier_pipeline(epochs=10, batch_size=16, processed_dir=processed_dir)

### Step 7: Run Evaluation Comparison Study
Evaluates both models on the held-out test split, generates the quantitative comparisons table, plots comparisons (ROC, confusion matrices, training histories, class distributions), performs t-SNE projection of real vs generated features, and compiles `final_research_report.md`.

In [ ]:
run_evaluation_study(processed_dir=processed_dir)

### Step 8: View Comparative Visualizations
Let's display the saved comparative curves, confusion matrices, t-SNE feature projection, and class distribution barchart.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(plt.imread('outputs/plots/confusion_matrix_comparison.png'))
axes[0].axis('off')
axes[0].set_title('Confusion Matrix Comparison')

axes[1].imshow(plt.imread('outputs/plots/performance_comparison.png'))
axes[1].axis('off')
axes[1].set_title('ROC Curve Comparison Study')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(plt.imread('outputs/plots/tsne_real_vs_generated.png'))
axes[0].axis('off')
axes[0].set_title('t-SNE Distribution Space')

axes[1].imshow(plt.imread('outputs/plots/class_distribution_comparison.png'))
axes[1].axis('off')
axes[1].set_title('Training Class Distribution')
plt.show()